In [ ]:
import functools
import warnings

import botocore
import boto3
from iterpop import iterpop as ip
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

from dishpylib.pyhelpers import fit_control_t_distns

warnings.filterwarnings("ignore")


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-06-18-multistint-complexity"


In [ ]:
@functools.lru_cache
def get_control_t_distns( bucket, prefix, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/{prefix}control-competitions/stage={2 + bool(prefix)}+what=collated/stint={stint}/',
    )

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )

    return fit_control_t_distns(control_df[
        control_df["Root ID"] == 1
    ].copy())


In [ ]:
def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    print(len(competitions_df), "competitions to preprocess")
    print(len(control_fits_df), "control fits available")
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: scipy_stats.t.cdf(
            row["Fitness Differential"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 40
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 40)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
prefix = ""
dfs = []
for bucket, stint in [
    ("prq49", 20),
    ("prq49", 40),
    ("prq49-20stint", 0),
    ("prq49-40stint", 0),
]:
    endeavor = {
        "prq49": 16,
        "prq49-20stint": 17,
        "prq49-40stint": 18,
    }[bucket]

    if "step" in bucket:
        step = int(bucket.split("-step")[-1]) + 1
    else:
        step = 0
    if "restint" in bucket:
        kind = bucket.split("-")[3]
    else:
        kind = None
    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    try:
        series_profiles, = bucket_handle.objects.filter(
            Prefix=f'endeavor={endeavor}/{prefix}variant-competitions/stage=3+what=collated/stint={stint}/',
        )
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
        control_fits_df = get_control_t_distns(bucket, prefix, endeavor, stint)
        df = pd.read_csv(
            f's3://{bucket}/{series_profiles.key}',
            compression='xz',
        )
        # df = df[df["Competition Series"] == 16005]
        # df = df.groupby([
        #     "genome variation"
        # ]).mean(numeric_only=True).reset_index()
        df["Stint"] = stint
        df["Series"] = df["Competition Series"]
        dfdigest = "{:x}".format( hash_pandas_object( df ).sum() )
        df = preprocess_competition_fitnesses(df, control_fits_df)
        assert "Series" in df.columns, df.columns

        df = df.copy()
        df["bucket"] = bucket
        df["variant"] = {"cryptic-": "skeleton", "": "wildtype"}[prefix]
        dfs.append(df)
    except Exception as e:
        print(e)
        print(f"Skipping {bucket=}, {stint=}, {prefix=}")


In [ ]:
df = pd.concat(dfs)


In [ ]:
pd.options.display.max_columns = None


In [ ]:
dfxx = df[
    df["Root ID"] == 1
].groupby(["Series", "Stint", "bucket", "variant"]).agg(
    {
        "Is More Fit": "sum",
        "Is Less Fit": "sum",
        "Is Neutral": "sum",
        "genome variation": "count",
    },
)
dfxx


In [ ]:
dfxx = dfxx.reset_index(drop=False)


In [ ]:
dfxx["bucket-stint"] = dfxx["bucket"] + "-stint" + dfxx["Stint"].astype(str)


In [ ]:
for y in ["Is More Fit", "Is Less Fit"]:
    with tp.teed(
        sns.relplot,
        data=dfxx,
        x="Series",
        y=y,
        col="bucket-stint",
        markers=True,
        dashes=False,
        facet_kws=dict(sharex=False),
    ) as _:
        pass


In [ ]:
for y in ["Is More Fit", "Is Less Fit"]:
    with tp.teed(
        sns.violinplot,
        data=dfxx,
        y=y,
        hue="bucket-stint",
        x="bucket-stint",
        cut=0,
    ) as _:
        pass


In [ ]:
for y in ["Is More Fit", "Is Less Fit"]:
    with tp.teed(
        sns.stripplot,
        data=dfxx,
        y=y,
        hue="bucket-stint",
        x="bucket-stint",
    ) as _:
        pass
